# 🚁 Aegis OS — Session 4: Obstacle Avoidance
## The AI learns to weave through cylinder forests using 32-ray LiDAR

---

### ✅ Before You Start:
1. **Runtime → Change Runtime Type → T4 GPU** _(critical — do this FIRST)_
2. This notebook is **100% self-contained** — no zip upload needed!
3. All code is built into the cells below
4. Training runs for **5 hours 10 minutes** automatically

### 📋 Run Each Block ONE AT A TIME, Top to Bottom

---

### What Session 4 Teaches:
- **Input**: 32 synthetic LiDAR rays (0.0 = obstacle touching, 1.0 = clear) + 10-DOF physics
- **Goal**: Fly 500m to a target through a dense cylinder forest without crashing
- **Reward**: +distance_progress, -collision (-150), +reach_target (+500)
- **Starts from**: Session 3 brain (`session3_final.pth`) stored in your Google Drive

## 🔵 Block 1 — Check GPU
_If this shows NO GPU, stop and change runtime type first._

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'✅ GPU Active: {name}')
    print(f'   VRAM: {vram} GB')
    print(f'   PyTorch: {torch.__version__}')
    print()
    print('Ready for Session 4 training!')
else:
    print('❌ NO GPU DETECTED!')
    print('   Go to: Runtime → Change Runtime Type → T4 GPU')
    print('   Then reconnect and run this cell again.')

## 🔵 Block 2 — Mount Google Drive
_Your checkpoint will be saved here. It survives if Colab disconnects._

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/aegis_checkpoints/session_4', exist_ok=True)
os.makedirs('/content/drive/MyDrive/aegis_checkpoints/latest',    exist_ok=True)
print('✅ Google Drive mounted!')
print('   Checkpoint folder: My Drive/aegis_checkpoints/')

## 🔵 Block 3 — Verify Session 3 Brain
_This checks that your `session3_final.pth` is in Google Drive and ready to load._

In [ ]:
import torch, os

# Check common locations for the Session 3 brain
possible_paths = [
    '/content/drive/MyDrive/aegis_checkpoints/session3_final.pth',
    '/content/drive/MyDrive/aegis_checkpoints/latest/latest.pth',
    '/content/drive/MyDrive/session3_final.pth',
]

found_path = None
for p in possible_paths:
    if os.path.exists(p):
        found_path = p
        break

if found_path:
    c = torch.load(found_path, map_location='cpu', weights_only=False)
    print(f'✅ Brain found at: {found_path}')
    print(f'   Session:     {c.get("session", "?")}  (should be 3)')
    print(f'   Update:      {c.get("update", "?")}')
    print(f'   Total Steps: {c.get("total_steps", 0):,}')
    dim = c['actor']['physics_net.0.weight'].shape[1]
    print(f'   Physics Dim: {dim}  (10 = correct for Session 4!)')
    print()
    print('Block 3 complete. Session 3 brain ready!')
else:
    print('❌ Session 3 brain NOT FOUND!')
    print()
    print('Please upload session3_final.pth to Google Drive.')
    print('Expected location: My Drive/aegis_checkpoints/session3_final.pth')
    print()
    print('To upload:')
    print('  1. Open drive.google.com')
    print('  2. Navigate to aegis_checkpoints folder')
    print('  3. Upload session3_final.pth from your computer models/ folder')

## 🔵 Block 4 — Define Neural Network Architecture
_This defines the same VisionPPOActor/Critic used since Session 1. Must match exactly._

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Normal

class VisionPPOActor(nn.Module):
    """
    The same actor architecture used across all Aegis sessions.
    Input:  32-ray LiDAR depth scan  +  10-DOF physics state
    Output: 4 continuous actions (Pitch, Roll, Yaw, Thrust) in [-1, 1]
    """
    def __init__(self, depth_rays=32, physics_dim=10, action_dim=4):
        super().__init__()
        self.vision_net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * (depth_rays // 4), 64),
            nn.ReLU()
        )
        self.physics_net = nn.Sequential(
            nn.Linear(physics_dim, 64),
            nn.ReLU()
        )
        self.actor_net = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Tanh()
        )
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, depth_scan, physics_state):
        vis  = self.vision_net(depth_scan)
        phys = self.physics_net(physics_state)
        mean = self.actor_net(torch.cat([vis, phys], dim=1))
        std  = self.log_std.exp().expand_as(mean)
        return mean, std


class VisionPPOCritic(nn.Module):
    def __init__(self, depth_rays=32, physics_dim=10):
        super().__init__()
        self.vision_net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * (depth_rays // 4), 64),
            nn.ReLU()
        )
        self.physics_net = nn.Sequential(
            nn.Linear(physics_dim, 64),
            nn.ReLU()
        )
        self.critic_net = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, depth_scan, physics_state):
        vis  = self.vision_net(depth_scan)
        phys = self.physics_net(physics_state)
        return self.critic_net(torch.cat([vis, phys], dim=1))

print('✅ Neural network architecture defined.')
print(f'   VisionPPOActor  — inputs: (1,32) depth + (10,) physics → 4 actions')
print(f'   VisionPPOCritic — inputs: (1,32) depth + (10,) physics → 1 value')

## 🔵 Block 5 — Define Session 4 Training Environment
_A vectorized physics simulation with 2048 parallel drones. Each drone navigates through a random cylinder forest using synthetic 32-ray LiDAR — exactly matching the AirSim bridge._

In [ ]:
import torch
import math

class Session4ObstacleEnv:
    """
    Session 4: Obstacle Avoidance Training Environment
    ===================================================
    - 2048 drones fly in parallel (vectorized on GPU)
    - Each episode: drone spawns in a 100m clear zone, target is 500m ahead
    - 20 cylinders per drone, randomly placed between 100m–600m radius
    - 32 LiDAR rays computed analytically (ray-circle intersection math)
      — identical to the AirSim bridge synthetic LiDAR implementation
    - Altitude locked to 8m via pitch override (matches AirSim bridge)

    Reward structure:
      +5  * (1 / dist_to_target)       — continuous approach reward
      +0.1 * delta_dist_progress        — reward for closing distance each step
      -150 if collision with cylinder   — episode terminates
      +500 if reaches target (< 10m)   — SUCCESS, episode terminates
      -100 if out of bounds / timeout   — episode terminates
    """

    NUM_RAYS     = 32
    MAX_RANGE    = 50.0   # LiDAR max range (metres)
    NUM_OBS      = 20     # cylinders per environment
    OBS_RADIUS   = 1.5    # cylinder collision radius (metres)
    CRUISE_ALT   = 8.0    # metres above ground
    DT           = 0.05   # physics timestep
    MAX_STEPS    = 800    # max steps per episode

    def __init__(self, num_envs, device):
        self.num_envs = num_envs
        self.device   = device

        # Pre-compute ray angles (world-aligned; we rotate to body frame in raycast)
        angles = torch.linspace(0, 2 * math.pi, self.NUM_RAYS + 1, device=device)[:-1]
        self.ray_cos = torch.cos(angles)   # (32,)
        self.ray_sin = torch.sin(angles)   # (32,)

        self.pos          = torch.zeros(num_envs, 3, device=device)  # x, y, z
        self.vel          = torch.zeros(num_envs, 3, device=device)
        self.attitude     = torch.zeros(num_envs, 3, device=device)  # pitch, roll, yaw
        self.target       = torch.zeros(num_envs, 3, device=device)
        self.obs_xy       = torch.zeros(num_envs, self.NUM_OBS, 2, device=device)
        self.steps        = torch.zeros(num_envs, dtype=torch.long, device=device)
        self.prev_dist    = torch.zeros(num_envs, device=device)

        self.reset(torch.ones(num_envs, dtype=torch.bool, device=device))

    def reset(self, mask):
        n   = int(mask.sum().item())
        if n == 0:
            return
        idx = mask.nonzero(as_tuple=True)[0]

        # Drone spawns near origin at cruise altitude
        self.pos[idx, 0] = (torch.rand(n, device=self.device) - 0.5) * 10.0
        self.pos[idx, 1] = (torch.rand(n, device=self.device) - 0.5) * 10.0
        self.pos[idx, 2] = self.CRUISE_ALT
        self.vel[idx]    = 0.0
        self.attitude[idx] = 0.0
        self.steps[idx]  = 0

        # Target: 400–600m ahead in a random direction
        target_angle = torch.rand(n, device=self.device) * 2 * math.pi
        target_dist  = 400.0 + torch.rand(n, device=self.device) * 200.0
        self.target[idx, 0] = self.pos[idx, 0] + target_dist * torch.cos(target_angle)
        self.target[idx, 1] = self.pos[idx, 1] + target_dist * torch.sin(target_angle)
        self.target[idx, 2] = self.CRUISE_ALT

        # Obstacles: scattered between 100–600m from spawn, in the direction of the target
        spawn_x = self.pos[idx, 0].unsqueeze(1)  # (n, 1)
        spawn_y = self.pos[idx, 1].unsqueeze(1)
        r = 100.0 + torch.rand(n, self.NUM_OBS, device=self.device) * 500.0
        a = target_angle.unsqueeze(1) + (torch.rand(n, self.NUM_OBS, device=self.device) - 0.5) * 2.0
        self.obs_xy[idx, :, 0] = spawn_x + r * torch.cos(a)
        self.obs_xy[idx, :, 1] = spawn_y + r * torch.sin(a)

        dist_to_target = torch.norm(self.target[idx, :2] - self.pos[idx, :2], dim=1)
        self.prev_dist[idx] = dist_to_target

    def _raycast(self):
        """
        Analytically compute 32 LiDAR ray distances using ray-circle intersection.
        This is the EXACT same formula used in the AirSim bridge synthetic LiDAR.
        Returns: (num_envs, 1, 32) normalised depths in [0, 1]
        """
        yaw = self.attitude[:, 2]   # (E,)

        # Ray directions in world frame, rotated by drone yaw
        # ray_cos/sin: (32,), yaw: (E,) → broadcast to (E, 32)
        cos_yaw = torch.cos(yaw).unsqueeze(1)  # (E, 1)
        sin_yaw = torch.sin(yaw).unsqueeze(1)

        ray_dx = cos_yaw * self.ray_cos - sin_yaw * self.ray_sin  # (E, 32)
        ray_dy = sin_yaw * self.ray_cos + cos_yaw * self.ray_sin  # (E, 32)

        # Vector from drone to each obstacle centre: (E, N, 2)
        dx = self.obs_xy[:, :, 0] - self.pos[:, 0].unsqueeze(1)  # (E, N)
        dy = self.obs_xy[:, :, 1] - self.pos[:, 1].unsqueeze(1)  # (E, N)

        # Expand for broadcasting: (E, N, 1) and (E, 1, 32) → (E, N, 32)
        dx = dx.unsqueeze(2)     # (E, N, 1)
        dy = dy.unsqueeze(2)
        rdx = ray_dx.unsqueeze(1)  # (E, 1, 32)
        rdy = ray_dy.unsqueeze(1)

        # Ray-circle: dot = D·C, disc = dot² - (|C|² - r²)
        dot  = rdx * dx + rdy * dy                              # (E, N, 32)
        c_sq = dx * dx + dy * dy
        disc = dot * dot - (c_sq - self.OBS_RADIUS ** 2)

        hit   = disc >= 0
        t     = dot - torch.sqrt(disc.clamp(min=0))
        t     = torch.where(hit & (t > 0), t, torch.full_like(t, self.MAX_RANGE))
        # Take closest obstacle per ray: min over N dim
        depths, _ = t.min(dim=1)   # (E, 32)
        depths = depths.clamp(0, self.MAX_RANGE) / self.MAX_RANGE

        return depths.unsqueeze(1)   # (E, 1, 32)

    def _get_state(self):
        depth_scan = self._raycast()   # (E, 1, 32)

        dist_vec  = self.target - self.pos                           # (E, 3)
        norm_dist = torch.clamp(dist_vec / 50.0, -1.0, 1.0)
        norm_vel  = torch.clamp(self.vel  / 15.0, -1.0, 1.0)

        # physics_state: [pitch, roll, yaw, vx, vy, vz, dx, dy, dz, vehicle_type]
        vehicle_type = torch.zeros(self.num_envs, 1, device=self.device)  # 0 = drone
        physics = torch.cat([
            self.attitude,   # (E, 3)
            norm_vel,        # (E, 3)
            norm_dist,       # (E, 3)
            vehicle_type     # (E, 1)
        ], dim=1)            # (E, 10)

        return depth_scan, physics

    def step(self, actions):
        self.steps += 1

        # Actions: [pitch, roll, yaw_rate, thrust]  all in [-1, 1]
        pitch_cmd = actions[:, 0] * 0.5
        roll_cmd  = actions[:, 1] * 0.5
        yaw_rate  = actions[:, 2] * 1.0
        thrust    = (actions[:, 3] + 1.0) * 0.5 * 22.0

        # Altitude hold override — exactly as in AirSim bridge _send_action
        alt_error  = self.pos[:, 2] - self.CRUISE_ALT
        pitch_cmd  = pitch_cmd + torch.clamp(-alt_error * 0.15, -0.5, 0.5)

        # Update attitude
        self.attitude[:, 0] += (pitch_cmd - self.attitude[:, 0]) * 0.15
        self.attitude[:, 1] += (roll_cmd  - self.attitude[:, 1]) * 0.15
        self.attitude[:, 2] += yaw_rate * self.DT

        # Physics update
        cy = torch.cos(self.attitude[:, 2])
        sy = torch.sin(self.attitude[:, 2])
        cp = torch.cos(self.attitude[:, 0])
        sp = torch.sin(self.attitude[:, 0])
        cr = torch.cos(self.attitude[:, 1])

        accel_x = (cy * sp * cr + sy * self.attitude[:, 1]) * thrust
        accel_y = (sy * sp * cr - cy * self.attitude[:, 1]) * thrust
        accel_z = cp * cr * thrust - 9.81

        drag = -0.12 * self.vel
        self.vel[:, 0] += (accel_x + drag[:, 0]) * self.DT
        self.vel[:, 1] += (accel_y + drag[:, 1]) * self.DT
        self.vel[:, 2] += (accel_z + drag[:, 2]) * self.DT
        self.pos       += self.vel * self.DT

        # --- Collision detection ---
        obs_dx = self.obs_xy[:, :, 0] - self.pos[:, 0].unsqueeze(1)  # (E, N)
        obs_dy = self.obs_xy[:, :, 1] - self.pos[:, 1].unsqueeze(1)
        obs_dist = torch.sqrt(obs_dx**2 + obs_dy**2)                 # (E, N)
        collision = (obs_dist < self.OBS_RADIUS).any(dim=1)          # (E,)

        # --- Distance to target ---
        dist_xy   = torch.norm(self.target[:, :2] - self.pos[:, :2], dim=1)  # (E,)
        dist_3d   = torch.norm(self.target       - self.pos,         dim=1)
        reached   = dist_3d < 10.0                                           # (E,)

        # --- Out of bounds ---
        oob = (
            (self.pos[:, 2] > 80.0) |
            (self.pos[:, 2] < 0.0)  |
            (torch.abs(self.pos[:, 0]) > 1200.0) |
            (torch.abs(self.pos[:, 1]) > 1200.0)
        )

        # --- Timeout ---
        timeout = self.steps > self.MAX_STEPS

        # --- Rewards ---
        speed   = torch.norm(self.vel, dim=1)
        rewards = torch.full((self.num_envs,), -0.05, device=self.device)

        # Approach reward: closing in on target
        dist_progress = self.prev_dist - dist_xy
        rewards += dist_progress * 0.5

        # Proximity bonus
        rewards += 3.0 / (dist_3d + 1.0)

        # Action smoothness penalty
        rewards -= torch.norm(actions, dim=1) * 0.003

        # Collision penalty
        rewards = torch.where(collision, torch.full_like(rewards, -150.0), rewards)

        # Target reached bonus
        rewards = torch.where(reached, torch.full_like(rewards, +500.0), rewards)

        # Out of bounds penalty
        rewards = torch.where(oob, torch.full_like(rewards, -100.0), rewards)

        # Timeout penalty
        rewards = torch.where(timeout, torch.full_like(rewards, -50.0), rewards)

        self.prev_dist = dist_xy.clone()

        dones = collision | reached | oob | timeout
        n_reached = int(reached.sum().item())
        n_collision = int(collision.sum().item())

        if dones.any():
            self.reset(dones)

        next_depth, next_phys = self._get_state()
        return next_depth, next_phys, rewards, dones, n_reached, n_collision

print('✅ Session 4 Obstacle Avoidance Environment defined.')
print(f'   Parallel envs:  2048')
print(f'   LiDAR rays:     32 (identical to AirSim bridge)')
print(f'   Physics dim:    10 (matches session3_final.pth)')
print(f'   Obstacle count: {Session4ObstacleEnv.NUM_OBS} cylinders per drone')
print(f'   Safe zone:      100m (drones must build speed before encountering first obstacle)')

## 🔵 Block 6 — Define Checkpoint Manager & GAE
_Handles saving/loading of checkpoints._

In [ ]:
import torch, os, time, glob

class CheckpointManager:
    def __init__(self, checkpoint_dir, session):
        self.session_dir = os.path.join(checkpoint_dir, f'session_{session}')
        self.latest_dir  = os.path.join(checkpoint_dir, 'latest')
        self.session     = session
        os.makedirs(self.session_dir, exist_ok=True)
        os.makedirs(self.latest_dir,  exist_ok=True)

    def save(self, actor, critic, optimizer, update, total_steps, best_reward, extra=None):
        state = {
            'actor':       actor.state_dict(),
            'critic':      critic.state_dict(),
            'optimizer':   optimizer.state_dict(),
            'update':      update,
            'total_steps': total_steps,
            'best_reward': best_reward,
            'session':     self.session,
            'timestamp':   time.time(),
        }
        if extra:
            state.update(extra)
        fname = os.path.join(self.session_dir, f'checkpoint_update_{update:05d}.pth')
        torch.save(state, fname)
        torch.save(state, os.path.join(self.latest_dir, 'latest.pth'))
        print(f'   💾 Saved update {update:,} | steps {total_steps:,}')
        return fname

    def load_latest(self, actor, critic, optimizer, device):
        # First try to resume a Session 4 checkpoint
        s4_pths = sorted(glob.glob(os.path.join(self.session_dir, '*.pth')), key=os.path.getmtime)
        if s4_pths:
            path = s4_pths[-1]
            resuming = True
        else:
            # Fall back to Session 3 brain (from latest/ or session3_final.pth)
            possible = [
                '/content/drive/MyDrive/aegis_checkpoints/session3_final.pth',
                '/content/drive/MyDrive/aegis_checkpoints/latest/latest.pth',
                '/content/drive/MyDrive/session3_final.pth',
            ]
            path = next((p for p in possible if os.path.exists(p)), None)
            resuming = False

        if path is None or not os.path.exists(path):
            print('   ⚠️  No checkpoint found. Starting from scratch (random weights).')
            return 0, 0, float('-inf')

        print(f'   Loading from: {path}')
        state = torch.load(path, map_location=device, weights_only=False)

        def graft(model_state, old_state):
            for k, old_t in old_state.items():
                if k not in model_state:
                    continue
                new_t = model_state[k]
                if old_t.shape == new_t.shape:
                    new_t.copy_(old_t)
                elif k == 'physics_net.0.weight' and old_t.shape[1] == 9 and new_t.shape[1] == 10:
                    new_t[:, :9] = old_t
                    new_t[:, 9]  = 0.0
                    print(f'     [Surgery] Expanded physics_dim 9→10 for key {k}')
            return model_state

        actor.load_state_dict(graft(actor.state_dict(), state['actor']))
        critic.load_state_dict(graft(critic.state_dict(), state['critic']))

        total_steps = state.get('total_steps', 0)
        best_reward = state.get('best_reward', float('-inf'))

        if resuming:
            optimizer.load_state_dict(state['optimizer'])
            start_update = state.get('update', 0) + 1
            prev_session = state.get('session', 4)
            print(f'   ✅ Resumed Session {prev_session} at update {start_update-1:,} | steps {total_steps:,}')
        else:
            prev_session = state.get('session', 3)
            start_update = 0
            best_reward  = float('-inf')
            print(f'   ✅ Loaded Session {prev_session} brain → starting Session 4 from update 0')
            print(f'      Prior steps: {total_steps:,}')

        return start_update, total_steps, best_reward


def compute_gae(next_value, rewards, masks, values, gamma=0.99, lam=0.95):
    values  = values + [next_value]
    gae     = 0
    returns = []
    for step in reversed(range(len(rewards))):
        delta = rewards[step] + gamma * values[step+1] * masks[step] - values[step]
        gae   = delta + gamma * lam * masks[step] * gae
        returns.insert(0, gae + values[step])
    return returns

print('✅ CheckpointManager and GAE utility defined.')

## 🟢 Block 7 — START SESSION 4 TRAINING

**This cell runs for 5 hours 10 minutes. Do not close the browser tab.**

Expected output (first few updates):
```
 Update |        Steps |   Avg Rew | Targets | Crashes |     Loss | Elapsed |      ETA
      0 |    1,048,576 |    -23.45 |    0.00 |   42.31 |   2.3401 |   0.5m  |  309.5m
     25 |   26,214,400 |    -18.20 |    0.12 |   38.77 |   1.8902 |  12.1m  |  297.9m
     50 |   52,428,800 |     -9.45 |    0.45 |   28.10 |   1.3241 |  24.8m  |  285.2m
```
**Targets/Env** rising = AI learning to reach goal while avoiding obstacles ✅

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import time

# ── Hyperparameters ──────────────────────────────────────────────────────────
SESSION       = 4
CHECKPOINT_DIR = '/content/drive/MyDrive/aegis_checkpoints/'
MAX_HOURS     = 5.1667          # 5 hours 10 minutes
MAX_SECONDS   = MAX_HOURS * 3600

NUM_ENVS      = 2048            # parallel environments (T4 has 16GB VRAM — fits comfortably)
PPO_STEPS     = 512             # steps collected per update
PPO_EPOCHS    = 4               # PPO training epochs per update
MINI_BATCH    = 4096            # mini-batch size
MAX_UPDATES   = 1500            # max total updates this session
SAVE_EVERY    = 25              # save checkpoint every N updates
CLIP_EPS      = 0.2
GAMMA         = 0.99
LAM           = 0.95
LR            = 2e-4
ENTROPY_COEF  = 0.01

# ── Setup ────────────────────────────────────────────────────────────────────
t_start = time.time()
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('=' * 72)
print('  Aegis OS — Session 4: Obstacle Avoidance Training')
print(f'  Device:    {device}   ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM:      {vram:.1f} GB')
print(f'  Max time:  {MAX_HOURS:.2f} hours (5h 10m)')
print(f'  Environments: {NUM_ENVS:,}  |  Steps/update: {NUM_ENVS * PPO_STEPS:,}')
print(f'  Saves to:  {CHECKPOINT_DIR}')
print('=' * 72)
print()

# ── Build models ─────────────────────────────────────────────────────────────
actor     = VisionPPOActor(depth_rays=32, physics_dim=10, action_dim=4).to(device)
critic    = VisionPPOCritic(depth_rays=32, physics_dim=10).to(device)
optimizer = optim.Adam(
    list(actor.parameters()) + list(critic.parameters()),
    lr=LR, eps=1e-5
)

# ── Load checkpoint ───────────────────────────────────────────────────────────
ckpt_mgr = CheckpointManager(CHECKPOINT_DIR, SESSION)
start_update, total_steps, best_reward = ckpt_mgr.load_latest(
    actor, critic, optimizer, device
)

scheduler = optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1.0,
    end_factor=0.1,
    total_iters=max(1, MAX_UPDATES - start_update)
)

# ── Build environment ─────────────────────────────────────────────────────────
env = Session4ObstacleEnv(num_envs=NUM_ENVS, device=device)
depth, phys = env._get_state()

print()
print(f'  Starting from update {start_update:,}. Let the training begin!')
print()
print(f'  {"Update":>7} | {"Steps":>12} | {"Avg Rew":>9} | {"Targets":>7} | {"Crashes":>7} | {"Loss":>8} | {"Elapsed":>7} | {"ETA":>8}')
print(f'  {"-" * 80}')

last_update = start_update
timed_out   = False

for update in range(start_update, MAX_UPDATES):
    elapsed = time.time() - t_start
    if elapsed > MAX_SECONDS:
        print(f'\n  ⏰ Time limit ({MAX_HOURS:.2f}h). Saving final checkpoint...')
        ckpt_mgr.save(actor, critic, optimizer, update, total_steps, best_reward)
        last_update = update
        timed_out   = True
        break

    # ── Collect rollout ───────────────────────────────────────────────────────
    log_probs_buf, values_buf = [], []
    depths_buf,   phys_buf   = [], []
    actions_buf,  rewards_buf, masks_buf = [], [], []
    targets_this   = 0
    collisions_this = 0

    actor.eval()
    critic.eval()
    with torch.no_grad():
        for _ in range(PPO_STEPS):
            depths_buf.append(depth)
            phys_buf.append(phys)

            mean, std = actor(depth, phys)
            dist_     = Normal(mean, std)
            action    = torch.clamp(dist_.sample(), -1.0, 1.0)
            value     = critic(depth, phys)

            next_depth, next_phys, reward, done, n_reach, n_crash = env.step(action)
            targets_this    += n_reach
            collisions_this += n_crash

            log_probs_buf.append(dist_.log_prob(action).sum(-1, keepdim=True))
            values_buf.append(value)
            rewards_buf.append(reward.unsqueeze(1))
            masks_buf.append((~done).float().unsqueeze(1))
            actions_buf.append(action)
            depth, phys = next_depth, next_phys

        next_value = critic(depth, phys)

    # ── Compute advantages ────────────────────────────────────────────────────
    returns    = compute_gae(next_value, rewards_buf, masks_buf, values_buf, GAMMA, LAM)
    returns    = torch.cat(returns).detach()
    log_probs  = torch.cat(log_probs_buf).detach()
    values_old = torch.cat(values_buf).detach()
    states_d   = torch.cat(depths_buf)
    states_p   = torch.cat(phys_buf)
    acts       = torch.cat(actions_buf)

    advantage  = returns - values_old
    advantage  = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

    # ── PPO update ────────────────────────────────────────────────────────────
    actor.train()
    critic.train()
    total_loss = 0.0
    batch_size = NUM_ENVS * PPO_STEPS

    for epoch in range(PPO_EPOCHS):
        perm = torch.randperm(batch_size, device=device)
        for start in range(0, batch_size, MINI_BATCH):
            idx = perm[start: start + MINI_BATCH]

            mean_new, std_new = actor(states_d[idx], states_p[idx])
            dist_new      = Normal(mean_new, std_new)
            new_log_probs = dist_new.log_prob(acts[idx]).sum(-1, keepdim=True)
            entropy       = dist_new.entropy().mean()

            ratio  = torch.exp(new_log_probs - log_probs[idx])
            surr1  = ratio * advantage[idx]
            surr2  = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantage[idx]
            a_loss = -torch.min(surr1, surr2).mean()

            v_pred = critic(states_d[idx], states_p[idx])
            c_loss = (returns[idx] - v_pred).pow(2).mean()

            loss   = a_loss + 0.5 * c_loss - ENTROPY_COEF * entropy
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(actor.parameters()) + list(critic.parameters()), 0.5
            )
            optimizer.step()

    scheduler.step()

    # ── Logging ───────────────────────────────────────────────────────────────
    total_steps    += NUM_ENVS * PPO_STEPS
    avg_reward      = torch.cat(rewards_buf).mean().item()
    n_batches       = PPO_EPOCHS * (batch_size // MINI_BATCH)
    avg_loss        = total_loss / max(1, n_batches)
    elapsed_m       = elapsed / 60.0
    updates_done    = update - start_update + 1
    time_per_upd    = elapsed / max(1, updates_done)
    eta_m           = ((MAX_UPDATES - update - 1) * time_per_upd) / 60.0
    target_rate     = targets_this / NUM_ENVS
    crash_rate      = collisions_this / NUM_ENVS

    if avg_reward > best_reward:
        best_reward = avg_reward

    last_update = update

    if update % 5 == 0:
        print(f'  {update:>7,} | {total_steps:>12,} | {avg_reward:>9.2f} | '
              f'{target_rate:>7.2f} | {crash_rate:>7.2f} | {avg_loss:>8.4f} | '
              f'{elapsed_m:>5.1f}m | {eta_m:>6.1f}m')

    if (update + 1) % SAVE_EVERY == 0:
        ckpt_mgr.save(actor, critic, optimizer, update, total_steps, best_reward)

# ── Final summary ──────────────────────────────────────────────────────────
elapsed_h = (time.time() - t_start) / 3600
print()
print('=' * 72)
if timed_out:
    print(f'  ⏰ Session 4 paused at update {last_update:,} / {MAX_UPDATES:,}')
    print(f'  Resume anytime — run Block 7 again on a fresh Colab session!')
else:
    ckpt_mgr.save(actor, critic, optimizer, last_update, total_steps, best_reward)
    print(f'  ✅ Session 4 Training COMPLETE! ({last_update:,} updates)')
print(f'  Total env steps:  {total_steps:,}')
print(f'  Best avg reward:  {best_reward:.4f}')
print(f'  Elapsed:          {elapsed_h:.2f} hours')
print('=' * 72)

## 🔵 Block 8 — Save Final Brain & Download
_Run this AFTER training completes. Saves as `session4_final.pth` and downloads it._

In [ ]:
import torch, time, os, shutil
from google.colab import files

src  = '/content/drive/MyDrive/aegis_checkpoints/latest/latest.pth'
dest = '/content/drive/MyDrive/aegis_checkpoints/session4_final.pth'

shutil.copy(src, dest)
print(f'✅ Saved: {dest}')

# Also copy locally so we can download directly
local_path = '/content/session4_final.pth'
shutil.copy(src, local_path)

c = torch.load(dest, map_location='cpu', weights_only=False)
print()
print('=== SESSION 4 FINAL RESULTS ===')
print(f'Session:     {c.get("session")}')
print(f'Update:      {c.get("update"):,} / 1500')
print(f'Total Steps: {c.get("total_steps", 0):,}')
print(f'Best Reward: {c.get("best_reward", 0):.4f}')
dim = c['actor']['physics_net.0.weight'].shape[1]
print(f'Physics Dim: {dim}  (10 = correct)')
saved = time.strftime('%Y-%m-%d %H:%M', time.localtime(c.get('timestamp', 0)))
print(f'Saved at:    {saved}')

print()
print('Downloading session4_final.pth to your computer...')
files.download(local_path)

print()
print('NEXT STEPS:')
print('  1. Move session4_final.pth to your models/ folder on your computer')
print('  2. Run: python scripts/airsim_bridge.py --model models/session4_final.pth')